In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 07:57:14.881742: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 07:57:15.549059: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 07:57:17,374 [DEBUG] [LogService] Rain is initialized
2023-07-02 07:57:17,374 [DEBUG] [LogService] Creating coordinator
2023-07-02 07:57:17,375 [DEBUG] [LogService] Coordinator is initialized
2023-07-02 07:57:17,375 [DEBUG] [LogService] LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_sync()

2023-07-02 07:57:17,382 [DEBUG] [LogService] Creating workers
2023-07-02 07:57:17,387 [INFO] [LogService] provisioner is serving
2023-07-02 07:57:17,388 [DEBUG] [LogService] Starting coordinator
2023-07-02 07:57:17,389 [INFO] [LogService] coordinator is serving
2023-07-02 07:57:17,390 [DEBUG] [LogService] sending the num of workers to the provisioner
2023-07-02 07:57:17,394 [DEBUG] [LogService] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 07:57:17,396 [DEBUG] [LogService] sent Success receiving the number of workers to the provisioner
2023-07-02 07:57:17,397 [DEBUG] [LogService] Creating 3 workers
2023-07-02 07:57:17,398 [INFO] [LogService] Worker is running on port: 50151
2023-07-02 07:57:17,400 [INFO] [LogService] Worker is running on port: 50152
2023-07-02 07:57:17,403 [INFO] [LogService] Worker is running on port: 50153
2023-07-02 07:57:17,404 [DEBUG] [LogService] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: 

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 5ms/step - loss: 0.7007 - accuracy: 0.7821
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.7045 - accuracy: 0.7752
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.7089 - accuracy: 0.7774
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.2983 - accuracy: 0.9090
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.3046 - accuracy: 0.9085
sending data to coordinator


2023-07-02 07:57:52,775 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:57:52,777 [INFO] [LogService] thread 1 is done
2023-07-02 07:57:52,778 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:57:52,803 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:57:52,869 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/1_1_trained.pkl in coordinator
2023-07-02 07:57:52,871 [INFO] [LogService] thread 2 is done
2023-07-02 07:57:52,946 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/2_1_trained.pkl in coordinator
2023-07-02 07:57:52,947 [INFO] [LogService] thread 3 is done
2023-07-02 07:57:53,022 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/3_1_trained.pkl in coordinator
2023-07-02 07:57:53,091 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:57:53,163 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:5

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 1s 6ms/step - loss: 0.2562 - accuracy: 0.9232
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 1s 6ms/step - loss: 0.2547 - accuracy: 0.9235
Epoch 2/2
157/157 [==============================] - 1s 6ms/step - loss: 0.1954 - accuracy: 0.9406
sending data to coordinator
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.1914 - accuracy: 0.9427
sending data to coordinator


2023-07-02 07:58:11,403 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:58:11,411 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:58:11,413 [INFO] [LogService] thread 1 is done
2023-07-02 07:58:11,445 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:58:11,490 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/1_2_trained.pkl in coordinator
2023-07-02 07:58:11,491 [INFO] [LogService] thread 2 is done
2023-07-02 07:58:11,560 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/2_2_trained.pkl in coordinator
2023-07-02 07:58:11,561 [INFO] [LogService] thread 3 is done
2023-07-02 07:58:11,631 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/3_2_trained.pkl in coordinator
2023-07-02 07:58:11,696 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:58:11,769 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:5

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1903 - accuracy: 0.9431
Epoch 2/2
157/157 [==============================] - 2s 7ms/step - loss: 0.1728 - accuracy: 0.9487
Epoch 2/2
157/157 [==============================] - 1s 5ms/step - loss: 0.1575 - accuracy: 0.9535
sending data to coordinator
sending data to coordinator
157/157 [==============================] - 1s 5ms/step - loss: 0.1457 - accuracy: 0.9557
sending data to coordinator


2023-07-02 07:58:30,517 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:58:30,524 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:58:30,525 [INFO] [LogService] thread 1 is done
2023-07-02 07:58:30,570 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:58:30,596 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/1_3_trained.pkl in coordinator
2023-07-02 07:58:30,597 [INFO] [LogService] thread 2 is done
2023-07-02 07:58:30,667 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/2_3_trained.pkl in coordinator
2023-07-02 07:58:30,667 [INFO] [LogService] thread 3 is done
2023-07-02 07:58:30,740 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/3_3_trained.pkl in coordinator
2023-07-02 07:58:30,804 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:58:31,039 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:5

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0977 - accuracy: 0.9707

Test accuracy: 97.1%
